# Plan step 4: Translate the gold chunks into their twins (Mistral via Ollama)

Every fact gets a **twin**: its gold chunk translated into the other language
(German chunk → English twin, English chunk → German twin). `twin_of` links a twin to the chunk it was translated from.
Which version ends up in the index (German, English or both) is decided later, in plan step 5 (conditions).

**Which facts:** every question in `data/questions/questions.jsonl` that is **not** marked `no` in
`review/03_questions.csv`. An empty `approved` cell counts as included (pilot). Question, answer and key fact
are taken from the review sheet, so your edits there are used.

**How:** the chunk is translated a few sentences at a time (so nothing gets dropped), then checked:

| Check | Why |
|---|---|
| length of each piece within 0.6–1.5 × the original (in words) | nothing dropped or invented |
| ≥ 90 % of the original's numbers appear in the translation | names/dates/numbers kept |
| written in the target language | no untranslated pieces |
| the parallel answer appears in the twin (word endings may differ) | the fact survived the translation |
| ≥ 70 % of the parallel question's content words appear in the twin | question wording close to the twin's wording |

If one of the last two fails, the parallel question/answer/key fact are **rewritten in the twin's own wording**
(same meaning) and checked again. Failed pieces are re-translated with the reason, up to 3 times.

**Output:** `data/twins/twins.jsonl`, `data/twins/rejected.jsonl`, review sheet `review/04_twins.csv`.

In [ ]:
import collections
import json
import re
import sys
import time
from pathlib import Path

import pandas as pd
import requests

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "ingest"))
from checks import (OTHER, contains, contains_loosely, find_span, is_language, language_counts,
                    number_coverage, numbers, split_sentences, word_overlap)

QUESTIONS_PATH = ROOT / "data" / "questions" / "questions.jsonl"
QUESTION_REVIEW = ROOT / "review" / "03_questions.csv"
ALL_CHUNKS = ROOT / "data" / "chunks" / "chunks.jsonl"
OUT_DIR = ROOT / "data" / "twins"
TWINS_PATH = OUT_DIR / "twins.jsonl"
REJECTED_PATH = OUT_DIR / "rejected.jsonl"
REVIEW_PATH = ROOT / "review" / "04_twins.csv"

OLLAMA = "http://localhost:11434"
MODEL = "mistral"
TEMPERATURE = 0.0          # translation: as literal and repeatable as possible
SEED = 42
MAX_ATTEMPTS = 3           # per piece, and for rewriting the parallel question
PIECE_WORDS = 80           # translate about this many words at a time

MIN_LENGTH_RATIO, MAX_LENGTH_RATIO = 0.6, 1.5
MIN_NUMBER_COVERAGE = 0.9
MIN_WORD_OVERLAP = 0.7

LANG_NAME = {"DE": "German", "EN": "English"}
print("Repository root:", ROOT)

In [ ]:
try:
    tags = requests.get(f"{OLLAMA}/api/tags", timeout=5).json()
except requests.ConnectionError:
    raise SystemExit("Ollama is not running. Start the Ollama app or run `ollama serve`.")
installed = [m["name"] for m in tags.get("models", [])]
print("Installed models:", installed)
assert any(n.split(":")[0] == MODEL.split(":")[0] for n in installed), f"Run: ollama pull {MODEL}"

## Load the included facts

Rows marked `no` (or `n`, `nein`) in the question review sheet are left out; everything else is included.

In [ ]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

TEXT_FIELDS = [f"{f}_{l}" for l in ("de", "en") for f in ("question", "answer", "key_fact")]

questions = pd.DataFrame(read_jsonl(QUESTIONS_PATH))
sheet = pd.read_csv(QUESTION_REVIEW, dtype=str, keep_default_na=False, encoding="utf-8-sig",
                    sep=None, engine="python")  # also reads Excel's ';' separated files
sheet = sheet.set_index("fact_id")

# The review sheet is the source of truth for the texts (your edits) and for approval
for field in TEXT_FIELDS:
    edited = questions["fact_id"].map(sheet[field])
    questions[field] = edited.where(edited.notna() & (edited != ""), questions[field])
questions["approved"] = questions["fact_id"].map(sheet["approved"]).fillna("").str.strip().str.lower()
included = questions[~questions["approved"].isin(["no", "n", "nein"])].copy()

chunks = {c["chunk_id"]: c for c in read_jsonl(ALL_CHUNKS)}
print(f"{len(included)} of {len(questions)} facts included")
print(included.groupby("origin_lang").size().rename("facts"))

In [ ]:
TRANSLATE_SYSTEM = """You are a professional translator of Egyptology texts.
Translate the {source} text you are given into {target}.

Rules:
1. Translate everything, sentence by sentence. Leave nothing out, add nothing, do not summarise.
2. Keep all names of people, places, sites, objects and institutions, and all numbers and dates.
3. Use the usual {target} terms for technical words; do not leave {source} words untranslated unless they are names.
4. Output only the translation.

Reply only with JSON: {{"translation": "..."}}"""

TRANSLATE_RETRY = """Your translation did not pass these checks:
{problems}
Translate the same text again, following all rules. Reply only with JSON."""

REVISE_SYSTEM = """You adjust a test question so that it matches the wording of a {target} passage.
You get the passage and a {target} question with its answer and key fact. Rewrite question, answer and key_fact so that:
1. They keep exactly the same meaning and ask for the same fact.
2. They use the passage's own words wherever possible.
3. The answer is copied word for word from the passage.
4. The key_fact is one sentence in the passage's wording that contains the answer.
5. The question does not contain the answer and ends with a single '?'.

Reply only with JSON: {{"question": "...", "answer": "...", "key_fact": "..."}}"""

REVISE_USER = """Passage:
\"\"\"
{text}
\"\"\"

Question, answer and key fact:
{item}"""

REVISE_RETRY = """Your reply did not pass these checks:
{problems}
Try again, following all rules. Reply only with JSON."""

In [ ]:
class OllamaError(Exception):
    """Ollama could not answer this request (server error), even in plain JSON mode."""

def ask_model(messages, schema, seed):
    """Send the conversation to Ollama and return the raw reply and the parsed JSON.

    Some texts make Ollama fail with a 500 error in JSON-schema mode; the same request is then
    tried once more in plain JSON mode. If that fails too, OllamaError is raised with Ollama's
    message, and the caller treats it as a failed attempt instead of stopping the whole run."""
    errors = []
    for fmt in (schema, "json"):
        response = requests.post(f"{OLLAMA}/api/chat", timeout=600, json={
            "model": MODEL, "messages": messages, "stream": False, "format": fmt,
            "options": {"temperature": TEMPERATURE, "seed": seed},
        })
        if response.ok:
            raw = response.json()["message"]["content"]
            return raw, json.loads(raw)
        mode = "schema" if fmt is schema else "plain json"
        errors.append(f"{response.status_code} in {mode} mode: {response.text[:200]}")
        print(f"  Ollama error {errors[-1]}")
        time.sleep(5)
    raise OllamaError("; ".join(errors))

TRANSLATION_SCHEMA = {"type": "object", "properties": {"translation": {"type": "string"}},
                      "required": ["translation"]}
ITEM_SCHEMA = {"type": "object",
               "properties": {k: {"type": "string"} for k in ("question", "answer", "key_fact")},
               "required": ["question", "answer", "key_fact"]}

def bullet_list(problems):
    return "\n".join(f"- {p}" for p in problems)

def pieces(text):
    """Groups of whole sentences of about PIECE_WORDS words."""
    groups, current = [], []
    for sentence in split_sentences(text):
        if current and len(" ".join(current + [sentence]).split()) > PIECE_WORDS:
            groups.append(" ".join(current))
            current = []
        current.append(sentence)
    if current:
        groups.append(" ".join(current))
    return groups

def check_piece(original, translation, target):
    problems = []
    ratio = len(translation.split()) / max(len(original.split()), 1)
    if not MIN_LENGTH_RATIO <= ratio <= MAX_LENGTH_RATIO:
        problems.append(f"The translation has {ratio:.0%} of the original's length; translate everything, "
                        "leave nothing out and add nothing.")
    if number_coverage(original, translation) < MIN_NUMBER_COVERAGE:
        problems.append("Keep all numbers and dates of the original.")
    own, other = language_counts(translation, target)
    if own <= other:
        problems.append(f"Write the whole translation in {LANG_NAME[target]}.")
    return problems

def translate_piece(text, source, target):
    messages = [{"role": "system", "content": TRANSLATE_SYSTEM.format(source=LANG_NAME[source], target=LANG_NAME[target])},
                {"role": "user", "content": text}]
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            raw, reply = ask_model(messages, TRANSLATION_SCHEMA, SEED + attempt)
            translation = str(reply.get("translation", "")).strip()
            problems = check_piece(text, translation, target)
        except OllamaError as error:
            translation, problems = "", [f"Ollama failed on this text: {error}"]
            continue  # try again with the next seed; the failed request is not added to the conversation
        except json.JSONDecodeError as error:
            raw, translation, problems = "", "", [f"The reply was not valid JSON ({error})."]
        if not problems:
            return translation, attempt, []
        messages += [{"role": "assistant", "content": raw},
                     {"role": "user", "content": TRANSLATE_RETRY.format(problems=bullet_list(problems))}]
    return translation, attempt, problems

In [ ]:
def check_parallel(item, twin_text, target, original_question):
    """The parallel question must still be answered by the twin, in the twin's wording."""
    q, a, k = item["question"], item["answer"], item["key_fact"]
    problems = []
    if not contains_loosely(twin_text, a):
        problems.append("The answer must be copied from the passage.")
    elif contains(q, a):
        problems.append("The question must not contain the answer.")
    overlap = word_overlap(q, twin_text, target)
    if overlap < MIN_WORD_OVERLAP:
        problems.append(f"Only {overlap:.0%} of the question's content words come from the passage; "
                        "use the passage's own words.")
    if not q.endswith("?") or q.count("?") != 1:
        problems.append("The question must end with a single '?'.")
    if numbers(q) != numbers(original_question):
        problems.append("Keep the numbers of the question.")
    if not is_language(q, target):
        problems.append(f"Write in {LANG_NAME[target]}.")
    return problems, round(overlap, 2)

def revise_parallel(item, twin_text, target):
    """Rewrite the parallel question/answer/key fact in the twin's own wording."""
    messages = [{"role": "system", "content": REVISE_SYSTEM.format(target=LANG_NAME[target])},
                {"role": "user", "content": REVISE_USER.format(text=twin_text, item=json.dumps(item, ensure_ascii=False))}]
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            raw, reply = ask_model(messages, ITEM_SCHEMA, SEED + attempt)
            revised = {k: str(reply.get(k, "")).strip() for k in ("question", "answer", "key_fact")}
            problems, overlap = check_parallel(revised, twin_text, target, item["question"])
        except OllamaError as error:
            revised, problems, overlap = dict(item), [f"Ollama failed on this text: {error}"], 0
            continue
        except json.JSONDecodeError as error:
            raw, revised, problems, overlap = "", dict(item), [f"The reply was not valid JSON ({error})."], 0
        if not problems:
            span = find_span(revised["answer"], twin_text)
            if span:
                revised["answer"] = span
            return revised, overlap, attempt, []
        messages += [{"role": "assistant", "content": raw},
                     {"role": "user", "content": REVISE_RETRY.format(problems=bullet_list(problems))}]
    return revised, overlap, attempt, problems

## Run

Results are saved after every fact; re-running the cell continues where it left off.
Expect roughly 30–90 seconds per chunk with Mistral 7B on a laptop.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
done = set()
for path in (TWINS_PATH, REJECTED_PATH):
    if path.exists():
        done.update(r["fact_id"] for r in read_jsonl(path))

for n, fact in enumerate(included.sort_values(["origin_lang", "pool_rank"]).to_dict("records"), start=1):
    if fact["fact_id"] in done:
        continue
    started = time.time()
    source, target = fact["origin_lang"], OTHER[fact["origin_lang"]]
    chunk = chunks[fact["chunk_id"]]

    # 1. translate the chunk piece by piece
    translated, problems, piece_attempts = [], [], []
    for piece in pieces(chunk["text"]):
        translation, attempts, piece_problems = translate_piece(piece, source, target)
        translated.append(translation)
        piece_attempts.append(attempts)
        problems += piece_problems
    twin_text = " ".join(translated)

    # 2. the parallel question must fit the twin; otherwise rewrite it in the twin's wording
    t = target.lower()
    parallel = {"question": fact[f"question_{t}"], "answer": fact[f"answer_{t}"], "key_fact": fact[f"key_fact_{t}"]}
    parallel_problems, overlap = check_parallel(parallel, twin_text, target, parallel["question"])
    revised = False
    if not problems and parallel_problems:
        parallel, overlap, _, parallel_problems = revise_parallel(parallel, twin_text, target)
        revised = True
    problems += parallel_problems

    record = {
        "fact_id": fact["fact_id"],
        "chunk_id": f"{fact['chunk_id']}_twin_{t}",
        "twin_of": fact["chunk_id"],
        "doc_id": chunk["doc_id"],
        "source": chunk.get("source", ""),
        "language": target,
        "is_twin": True,
        "text": twin_text,
        "n_words": len(twin_text.split()),
        "length_ratio": round(len(twin_text.split()) / chunk["n_words"], 2),
        "number_coverage": round(number_coverage(chunk["text"], twin_text), 2),
        f"question_{t}": parallel["question"], f"answer_{t}": parallel["answer"], f"key_fact_{t}": parallel["key_fact"],
        "parallel_revised": revised,
        "parallel_overlap": overlap,
        "piece_attempts": piece_attempts,
        "model": MODEL, "temperature": TEMPERATURE, "seed": SEED,
    }
    if problems:
        record["problems"] = problems
        path = REJECTED_PATH
    else:
        path = TWINS_PATH
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    status = "ok" + (" (question rewritten)" if revised else "") if not problems else "REJECTED"
    print(f"[{n}/{len(included)}] {fact['fact_id']}: {status}, {time.time() - started:.0f}s")

In [ ]:
REVIEW_COLUMNS = ["approved", "notes", "fact_id", "origin_lang", "length_ratio", "number_coverage",
                  "parallel_revised", "question_de", "answer_de", "question_en", "answer_en",
                  "original_text", "twin_text"]

twins = pd.DataFrame(read_jsonl(TWINS_PATH))
facts = included.set_index("fact_id")
twins["origin_lang"] = twins["fact_id"].map(facts["origin_lang"])
twins["original_text"] = twins["twin_of"].map(lambda cid: chunks[cid]["text"])
twins["twin_text"] = twins["text"]
for field in ("question", "answer"):
    for lang in ("de", "en"):
        col = f"{field}_{lang}"
        # the twin's language comes from the twin record (possibly rewritten), the origin language from step 5
        from_twin = twins[col] if col in twins else pd.Series("", index=twins.index)
        twins[col] = from_twin.where(from_twin.notna() & (from_twin != ""), twins["fact_id"].map(facts[col]))
twins["approved"], twins["notes"] = "", ""

if REVIEW_PATH.exists():  # keep earlier approvals and notes
    previous = pd.read_csv(REVIEW_PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig",
                           sep=None, engine="python").set_index("fact_id")
    for col in ("approved", "notes"):
        twins[col] = twins["fact_id"].map(previous[col]).fillna("")

twins.sort_values(["origin_lang", "fact_id"])[REVIEW_COLUMNS].to_csv(REVIEW_PATH, index=False, encoding="utf-8-sig")
print(f"Wrote {len(twins)} rows to {REVIEW_PATH.relative_to(ROOT)}")

In [ ]:
rejected = pd.DataFrame(read_jsonl(REJECTED_PATH)) if REJECTED_PATH.exists() else pd.DataFrame(columns=["fact_id", "language", "problems"])
summary = pd.DataFrame({
    "twins": twins.groupby("language").size(),
    "rejected": rejected.groupby("language").size(),
    "question_rewritten": twins[twins["parallel_revised"]].groupby("language").size(),
}).fillna(0).astype(int)
summary["mean_length_ratio"] = twins.groupby("language")["length_ratio"].mean().round(2)
summary.index.name = "twin language"
summary

In [ ]:
if len(rejected):
    display(rejected.explode("problems").groupby("language")["problems"].value_counts().rename("count").to_frame())
else:
    print("No rejected twins.")

## Review, then push

`review/04_twins.csv` shows each original next to its twin. Mark `approved` **no** only if the twin is
not a faithful translation (something left out or added, wrong meaning, untranslated words) or if the
parallel question no longer fits the twin. Empty counts as included.

```
git add notebooks/06_translate_twins.ipynb data/twins review/04_twins.csv
git commit -m "Step 4: twin translations"
git push
```